# 최종 분석보고서의 검증된 근거표
종로구·마포구의 공개 상권자료 외식업 10종, 2023~2025년 12분기를 대상으로 기존 최종 실행 결과를 보고서용으로 정리한다. 회귀모형을 다시 추정하지 않으며, 원자료·최종 분석표·기존 결과를 수정하지 않는다. 네트워크 요청 없이 위에서 아래로 실행한다.

결과는 `reports/20261006/analysis_report/evidence/`에 5개 CSV와 검증·정의 JSON으로 저장한다. 매출 미제공은 결측으로 유지하고, 상권 중첩이 존재하는 자료의 매출 합계를 자치구 경제규모로 해석하지 않는다. 이 노트북의 기술통계 단위는 **상권×업종×분기**다.

## 1. 경로와 입력 무결성
현재 경로에서 프로젝트 루트를 찾는다. 저장된 08·09·10번 메타데이터의 입력·출력 해시를 먼저 검증한다. 이후 읽는 파일의 실행 전후 SHA-256도 기록해 보고서 생성이 기존 자료를 바꾸지 않았는지 확인한다.

In [ ]:
# 현재 구조 및 과거 고정 명세의 경로를 함께 해석합니다.
from pathlib import Path as _ProjectPath
import json as _bootstrap_json
_project_home = next(p for p in [_ProjectPath.cwd(), *_ProjectPath.cwd().parents]
                     if (p / 'environment/project_paths.ipynb').is_file())
_project_notebook = _bootstrap_json.loads((_project_home / 'environment/project_paths.ipynb').read_text(encoding='utf-8'))
for _project_cell in _project_notebook['cells']:
    if 'project-runtime' in _project_cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_project_cell['source']), 'environment/project_paths.ipynb', 'exec'), globals())

from pathlib import Path
import hashlib, json
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

ROOT = PROJECT_ROOT
REPORTS = project_path('reports/20261006')
OUT = REPORTS / 'analysis_report/evidence'
OUT.mkdir(parents=True, exist_ok=True)
KEY = ['area_id', 'industry_id', 'quarter']
input_records = {}
pd.set_option('display.max_columns', 20)

In [ ]:
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def register(relative, expected=None, verification='실행 전후 일치'):
    path = project_path(relative)
    actual = sha256(path)
    if expected is not None:
        assert actual == expected, f'고정 해시 불일치: {relative}'
    if relative not in input_records:
        input_records[relative] = {
            'source_id': f'S{len(input_records)+1:02d}', 'path': relative,
            'bytes': path.stat().st_size, 'sha256_before': actual,
            'expected_sha256': expected, 'verification': verification}
    elif expected is not None:
        input_records[relative].update(expected_sha256=expected, verification=verification)
    return path

def read_json(relative):
    return project_json_loads(register(relative).read_text(encoding='utf-8'))

def read_csv(relative):
    return pd.read_csv(register(relative), float_precision='round_trip',
                       dtype={name: str for name in KEY})

scope_meta = read_json('reports/20261006/08_food_scope_metadata.json')
fe_meta = read_json('reports/20261006/09_analysis_summary.json')
gee_meta = read_json('reports/20261006/10_gee_analysis_summary.json')

In [ ]:
register('dataset/analysis/final/food_industry_panel.csv', scope_meta['panel_sha256'], '08 패널 해시')
register('dataset/analysis/final/food_industry_catalog.csv', scope_meta['catalog_sha256'], '08 업종목록 해시')
register(fe_meta['input_file'], fe_meta['input_sha256'], '09 입력 해시')
for relative, expected in fe_meta['output_sha256'].items():
    register(relative, expected, '09 결과 해시')
for relative, expected in gee_meta['input_sha256'].items():
    register(relative, expected, '10 입력 해시')
for name, expected in gee_meta['output_sha256'].items():
    register(f'reports/20261006/{name}', expected, '10 결과 해시')
register('reports/20261006/09_industry_effects.csv', gee_meta['fe_input_sha256'], '10의 FE 비교 입력 해시')
register('reports/20261006/09_model_sample_membership.csv', gee_meta['fe_membership_sha256'], '10의 FE 표본 해시')
assert fe_meta['seoul_refresh_run_id'] == gee_meta['seoul_refresh_run_id'] == scope_meta['seoul_refresh_run_id']
assert fe_meta['kopis_collection_run_id'] == gee_meta['kopis_collection_run_id']
print(f'메타데이터 일치: {sum(v["expected_sha256"] is not None for v in input_records.values())}개 파일')

**해석:** 입력과 기존 통계 결과가 서로 같은 고정 스냅샷을 가리키는지를 검사한다. 해시 검증은 계산·파일의 동일성에 관한 확인이며, 관측자료의 선택 문제나 인과적 타당성을 보장하지 않는다.

## 2. 분석 범위와 서로 다른 분모
전체 패널의 매출 제공률과 실제 모형 표본을 구분한다. 10종 공통 관측의 원패널 분모는 전체 1,812개 상권·분기다. FE 후 분모는 적어도 한 업종이 모형에 남은 상권·분기 수이므로 두 비율은 같은 모집단의 비율이 아니다.

In [ ]:
panel = read_csv('dataset/analysis/final/food_industry_panel.csv')
coverage = read_csv('reports/20261006/08_industry_coverage.csv')
district_coverage = read_csv('reports/20261006/08_district_industry_coverage.csv')
common_source = read_csv('reports/20261006/08_common_sample_scope.csv')
store_source = read_csv('reports/20261006/08_store_size_sales_coverage.csv')
membership = read_csv('reports/20261006/09_model_sample_membership.csv')
effects = read_csv('reports/20261006/09_industry_effects.csv')
contrasts = read_csv('reports/20261006/09_activity_change_contrasts.csv')
support = read_csv('reports/20261006/09_activity_support.csv')
assert not panel.duplicated(KEY).any()
assert (len(panel), panel.area_id.nunique(), panel.industry_id.nunique(), panel.quarter.nunique()) == (18120, 151, 10, 12)
assert panel.total_sales.notna().sum() == 7661
assert panel.loc[panel.total_sales.notna(), 'total_sales'].ge(0).all()
assert panel['regular_store_count'].equals(panel['store_count'])

In [ ]:
main_keys = membership.query("scenario == 'main_500' and outcome == 'log_sales'")[KEY]
night_keys = membership.query("scenario == 'main_500' and outcome == 'night_share_pct'")[KEY]
assert main_keys.sort_values(KEY).reset_index(drop=True).equals(night_keys.sort_values(KEY).reset_index(drop=True))
main_panel = main_keys.merge(panel, on=KEY, how='left', validate='one_to_one')
observed = panel.loc[panel.total_sales.notna()].copy()
singleton = observed.groupby(['area_id','industry_id']).quarter.transform('nunique').eq(1)
assert singleton.sum() == 15
assert observed.loc[~singleton, KEY].sort_values(KEY).reset_index(drop=True).equals(main_keys.sort_values(KEY).reset_index(drop=True))
balanced_keys = membership.query("scenario == 'balanced_500' and outcome == 'log_sales'")[KEY]
raw_counts = panel.loc[panel.total_sales.notna()].groupby(['area_id', 'quarter']).industry_id.nunique()
fe_counts = main_panel.groupby(['area_id', 'quarter']).industry_id.nunique()
raw_common = int(raw_counts.eq(10).sum())
fe_common = int(fe_counts.eq(10).sum())
assert raw_common == int(common_source.iloc[0].all_10_industries_observed_area_quarters) == 171
assert (len(main_panel), main_panel.area_id.nunique(), fe_common, len(fe_counts)) == (7646, 134, 170, 1592)
assert (len(balanced_keys), balanced_keys[['area_id', 'industry_id']].drop_duplicates().shape[0]) == (7020, 585)
print(f'전체 {len(panel):,}행 → 매출 제공 {panel.total_sales.notna().sum():,}행 → 기본 FE {len(main_panel):,}행')
print(f'10종 공통: 원패널 {raw_common}/1,812 상권·분기, FE 후 {fe_common}/{len(fe_counts):,} 상권·분기')

**해석:** 기본 FE에서 제외된 15행은 해당 상권×업종이 한 분기만 관측되어 최소 2분기 조건을 만족하지 않는 행이다. 12분기 균형표본 7,020행은 585개 상권×업종 조합이며, 같은 585개 상권이나 10종 공통 상권을 뜻하지 않는다.

## 3. 업종별 제공률과 관측자료의 기술통계
업종별 기대 관측은 151상권×12분기인 1,812행이다. 매출 중앙값은 **제공된 상권·분기 매출**의 중앙값이다. 야간 비중은 `총매출 > 0`, `0 ≤ 21~24시 매출 ≤ 총매출`이고 두 값이 모두 제공된 동일 행에서 `야간매출 합 / 총매출 합 × 100`으로 계산한다. 이때 합계는 비중의 분자·분모일 뿐 지역 전체 경제규모가 아니다.

In [ ]:
industry_rows = []
for (industry_id, industry), part in panel.groupby(['industry_id', 'industry'], sort=True):
    sales = part.loc[part.total_sales.notna(), 'total_sales']
    valid = (part.total_sales.gt(0) & part.night_sales.ge(0)
             & part.night_sales.le(part.total_sales))
    numerator = float(part.loc[valid, 'night_sales'].sum(min_count=1))
    denominator = float(part.loc[valid, 'total_sales'].sum(min_count=1))
    industry_rows.append(dict(industry_id=industry_id, industry=industry,
        expected_rows=len(part), sales_observed_rows=len(sales),
        sales_coverage_pct=100*len(sales)/len(part),
        observed_sales_areas=part.loc[part.total_sales.notna(), 'area_id'].nunique(),
        median_observed_quarter_sales_won=float(sales.median()),
        valid_night_rows=int(valid.sum()), night_sales_numerator_won=numerator,
        total_sales_denominator_won=denominator,
        weighted_night_share_pct=100*numerator/denominator))
industry_summary = pd.DataFrame(industry_rows)
check = industry_summary.merge(coverage, on='industry_id', validate='one_to_one', suffixes=('', '_source'))
assert np.array_equal(check.sales_observed_rows, check.sales_value_rows)
assert np.allclose(check.sales_coverage_pct, check.sales_value_coverage_pct)
assert np.array_equal(check.observed_sales_areas, check.observed_sales_areas_source)
display(industry_summary[['industry', 'sales_observed_rows', 'expected_rows', 'sales_coverage_pct',
    'median_observed_quarter_sales_won', 'valid_night_rows', 'weighted_night_share_pct']].style.format(precision=2))

In [ ]:
lowest = industry_summary.loc[industry_summary.sales_coverage_pct.idxmin()]
highest = industry_summary.loc[industry_summary.sales_coverage_pct.idxmax()]
display(Markdown(f'**해석:** 매출 제공률은 {lowest.industry} {lowest.sales_coverage_pct:.1f}%부터 '
    f'{highest.industry} {highest.sales_coverage_pct:.1f}%까지 차이가 난다. '
    '관측되는 상권 구성이 달라 업종별 중앙값·야간 비중의 차이를 업종 자체의 효과로 읽을 수 없다.'))

## 4. 지역 및 전체점포 규모별 매출 제공
지역표는 서로 다른 기대 관측 행 수를 분모로 쓴다. 점포 규모표는 기존 `08_store_size_sales_coverage.csv`의 값을 그대로 보존하고, 패널에서 같은 결과가 재현되는지만 대조한다. 전체점포는 일반점포와 프랜차이즈점포의 합이다.

In [ ]:
district_summary = panel.groupby('district', sort=True).agg(
    area_count=('area_id', 'nunique'), expected_rows=('quarter', 'size'),
    sales_observed_rows=('total_sales', 'count')).reset_index()
district_summary['sales_missing_rows'] = district_summary.expected_rows-district_summary.sales_observed_rows
district_summary['sales_coverage_pct'] = 100*district_summary.sales_observed_rows/district_summary.expected_rows
district_fe = main_panel.groupby('district').agg(fe_rows=('quarter', 'size'), fe_areas=('area_id', 'nunique'))
district_summary = district_summary.merge(district_fe, on='district', validate='one_to_one')
source_district = district_coverage.groupby('district')[['expected_rows','sales_value_rows']].sum()
assert np.array_equal(district_summary.set_index('district').expected_rows, source_district.expected_rows)
assert np.array_equal(district_summary.set_index('district').sales_observed_rows, source_district.sales_value_rows)
display(district_summary.style.format({'sales_coverage_pct': '{:.2f}'}))

**해석:** 지역별 제공률의 차이는 해당 지역에서 관측되는 업종·상권 구성의 차이를 포함한다. 미제공 매출이 없다고 가정하거나 관측 합계를 지역 전체 매출로 확대하지 않는다.

In [ ]:
store_rebuilt = panel.groupby('total_store_size_group', sort=True).agg(
    rows=('quarter', 'size'), sales_observed=('total_sales', 'count')).reset_index()
store_rebuilt['sales_missing'] = store_rebuilt.rows-store_rebuilt.sales_observed
store_rebuilt['sales_coverage_pct'] = 100*store_rebuilt.sales_observed/store_rebuilt.rows
sort_store = lambda frame: frame.sort_values('total_store_size_group').reset_index(drop=True)
pd.testing.assert_frame_equal(sort_store(store_rebuilt), sort_store(store_source), check_dtype=False)
pd.testing.assert_frame_equal(sort_store(store_source),
    sort_store(pd.DataFrame(scope_meta['sales_coverage_by_total_store_size'])), check_dtype=False)
store_summary = store_source.copy()
order = {name: i for i, name in enumerate(['0', '1~2', '3~5', '6~10', '11~50', '51+', '미제공'])}
store_summary = store_summary.sort_values('total_store_size_group', key=lambda s: s.map(order)).reset_index(drop=True)
display(store_summary.style.format({'sales_coverage_pct': '{:.2f}'}))

**해석:** 전체점포 1~2개인 4,430행에는 매출이 제공되지 않았다. 3~5개 집단은 66.6%, 6~10개 집단은 92.5%, 11~50개 집단은 97.8%, 51개 이상 집단은 100% 제공된다. 이러한 선택적 미제공 때문에 관측 매출 표본이 전체 외식업을 대표한다고 볼 수 없다. 일반점포 0·프랜차이즈 양수인 행을 점포 부재로 해석하지 않으며, 매출 모집단이 검증되지 않았으므로 점포당 매출도 제시하지 않는다.

## 5. 공연 10개→20개 시나리오: 기존 추정 결과 재배열
기존 09번 시나리오 결과에 동일 기본모형의 표본 수·상권 수·원 p값·BH 보정 p값을 일대일로 연결한다. **20개 업종×결과 검정**을 한 가족으로 보정한 값을 그대로 사용한다. 95% 신뢰구간은 개별 구간이며 다중비교를 보정한 동시 구간이 아니다.

총매출 값은 `log(1+관측 공연 ID 수)`가 `log(21/11)`만큼 달라질 때의 **(총매출+1) 조건부 기하평균 상대차이(%)**다. 산술평균 매출 변화율이나 개입의 인과효과가 아니다. 야간 값은 21~24시 매출 비중의 **퍼센트포인트(%p) 차이**다. 등록 공연 ID 수는 실제 상연 회차·관객 수와 다르다.

In [ ]:
main_effects = effects.loc[effects.scenario.eq('main_500')].copy()
join_key = ['industry_id', 'industry', 'outcome']
fields = ['p_value', 'p_fdr_primary', 'nobs_industry', 'area_clusters_industry',
          'nobs_total', 'area_clusters_total']
results = contrasts.merge(main_effects[join_key+fields], on=join_key, validate='one_to_one')
results = results.merge(support[join_key+['areas_spanning_10_to_20']], on=join_key, validate='one_to_one')
assert len(results) == 20 and results[fields].notna().all().all()
# 새 추정 없이 기존 계수·구간의 변환 규칙만 검증한다.
for outcome, part in main_effects.groupby('outcome'):
    expected = part.sort_values('industry_id')[['coefficient','ci_low','ci_high']].to_numpy()*np.log(21/11)
    if outcome == 'log_sales':
        expected = 100*np.expm1(expected)
    actual = results.loc[results.outcome.eq(outcome)].sort_values('industry_id')[['estimate','ci_low','ci_high']].to_numpy()
    assert np.allclose(actual, expected, rtol=1e-12, atol=1e-12)
results['contrast_label'] = '500m 관측 등록 공연 ID 10개→20개'
results['confidence_interval_type'] = 'individual_95pct_not_multiplicity_adjusted'
sales_results = results.loc[results.outcome.eq('log_sales')].sort_values('industry_id').reset_index(drop=True)
night_results = results.loc[results.outcome.eq('night_share_pct')].sort_values('industry_id').reset_index(drop=True)

In [ ]:
show_columns = ['industry', 'estimate', 'ci_low', 'ci_high', 'p_value', 'p_fdr_primary',
                'nobs_industry', 'area_clusters_industry', 'areas_spanning_10_to_20']
display(sales_results[show_columns].style.format(precision=4))

**매출 해석:** 업종별 시나리오의 개별 95% 구간은 모두 0을 포함한다. 방향·크기는 탐색적 추정치로 읽어야 하며, 통계적으로 유의하지 않다는 사실은 효과가 없다는 증거가 아니다. 업종마다 관측 상권과 기간이 다르므로 업종 순위나 대표 외식업의 인과적 차이로 해석하지 않는다.

In [ ]:
display(night_results[show_columns].style.format(precision=4))

In [ ]:
raw_significant = results.loc[results.p_value.lt(.05), ['industry', 'outcome', 'estimate', 'p_value', 'p_fdr_primary']]
assert results.p_fdr_primary.lt(.05).sum() == 0
details = ' / '.join(f'{row.industry}: 원 p={row.p_value:.4f}, BH p={row.p_fdr_primary:.4f}'
    for row in raw_significant.itertuples())
display(Markdown(f'**야간 해석:** {details}. 개별 구간에서 나타나는 신호가 있어도 '
    '20개 탐색 검정의 BH 보정 후 5% 기준을 통과한 업종·결과는 없다. '
    '`areas_spanning_10_to_20`은 같은 상권의 최솟값이 10 이하이고 최댓값이 20 이상인 수이며, '
    '정확히 10개와 20개인 두 분기를 관측했다는 뜻은 아니다.'))

## 6. 보고서용 파일 저장과 최종 확인
CSV는 수치를 반올림하지 않고 저장하고, 화면 표시에서만 반올림한다. 제공률과 추정 결과의 분모·단위·제약은 JSON에 함께 보존한다. 모든 입력의 실행 후 해시가 실행 전 해시와 같은지 다시 확인한다.

In [ ]:
def source_id(relative):
    return input_records[relative]['source_id']

outputs = {
    '01_industry_summary.csv': industry_summary,
    '02_district_summary.csv': district_summary,
    '03_store_size_coverage.csv': store_summary,
    '04_sales_contrasts.csv': sales_results,
    '05_night_contrasts.csv': night_results}
source_map = {
    '01_industry_summary.csv': ['dataset/analysis/final/food_industry_panel.csv','reports/20261006/08_industry_coverage.csv'],
    '02_district_summary.csv': ['dataset/analysis/final/food_industry_panel.csv','reports/20261006/08_district_industry_coverage.csv','reports/20261006/09_model_sample_membership.csv'],
    '03_store_size_coverage.csv': ['reports/20261006/08_store_size_sales_coverage.csv'],
    '04_sales_contrasts.csv': ['reports/20261006/09_activity_change_contrasts.csv','reports/20261006/09_industry_effects.csv','reports/20261006/09_activity_support.csv'],
    '05_night_contrasts.csv': ['reports/20261006/09_activity_change_contrasts.csv','reports/20261006/09_industry_effects.csv','reports/20261006/09_activity_support.csv']}
for filename, frame in outputs.items():
    frame['source_id'] = ';'.join(source_id(relative) for relative in source_map[filename])
    frame.to_csv(OUT/filename, index=False, encoding='utf-8-sig')
assert len(outputs) == 5

In [ ]:
scope = {
    'analysis_unit': 'area_id × industry_id × quarter', 'districts': ['종로구','마포구'],
    'area_count': 151, 'industry_count': 10, 'quarters': sorted(panel.quarter.unique().tolist()),
    'expected_rows': len(panel), 'sales_observed_rows': int(panel.total_sales.notna().sum()),
    'sales_missing_rows': int(panel.total_sales.isna().sum()),
    'sales_coverage_pct': float(100*panel.total_sales.notna().mean()),
    'fe_main_rows': len(main_panel), 'fe_main_area_clusters': int(main_panel.area_id.nunique()),
    'fe_singleton_rows_removed': int(panel.total_sales.notna().sum()-len(main_panel)),
    'raw_common_all_10_area_quarters': raw_common, 'raw_common_denominator_all_area_quarters': 1812,
    'raw_common_pct': 100*raw_common/1812,
    'fe_common_all_10_area_quarters': fe_common,
    'fe_common_denominator_area_quarters_with_any_retained_industry': len(fe_counts),
    'fe_common_pct_of_any_retained_industry': 100*fe_common/len(fe_counts),
    'fe_common_pct_of_full_panel_area_quarters': 100*fe_common/1812,
    'common_denominators_not_interchangeable': True,
    'balanced_12q_rows': len(balanced_keys), 'balanced_12q_area_industry_entities': 585,
    'primary_bh_family_size': 20, 'primary_bh_significant_count': 0,
    'seoul_refresh_run_id': fe_meta['seoul_refresh_run_id'],
    'kopis_collection_run_id': fe_meta['kopis_collection_run_id'],
    'new_model_fits': 0, 'network_requests': 0}
(OUT/'scope_summary.json').write_text(json.dumps(scope,ensure_ascii=False,indent=2),encoding='utf-8')

In [ ]:
definitions = {
    'median_observed_quarter_sales_won': '매출이 제공된 상권×업종×분기 총매출(원)의 중앙값; 전체 업종 경제규모가 아님',
    'weighted_night_share_pct': '유효한 동일 행의 21~24시 매출 합 / 총매출 합 × 100; 서로 다른 분자·분모 표본을 사용하지 않음',
    'night_sales_numerator_won': '가중 야간비중 계산용 분자; 중첩 상권을 제거한 경제규모 합계가 아님',
    'total_sales_denominator_won': '가중 야간비중 계산용 분모; 중첩 상권을 제거한 경제규모 합계가 아님',
    'sales_results_estimate': '(총매출+1) 조건부 기하평균 상대차이 % = 100×expm1(기존 FE 기울기×log(21/11)); 산술평균 변화율·인과효과가 아님',
    'night_results_estimate': '21~24시 매출 비중 차이 %p = 기존 FE 기울기×log(21/11)',
    'p_fdr_primary': '기본 500m의 10업종×2결과=20개 탐색 검정에 대한 기존 BH 보정 p값',
    'ci_low_ci_high': '기존 기본 FE의 개별 95% 신뢰구간을 해당 단위로 변환; 다중비교 동시 구간이 아님',
    'nobs_industry': '해당 업종·결과의 FE 관측 행 수',
    'area_clusters_industry': '해당 업종·결과의 FE 상권 수',
    'nobs_total_area_clusters_total': '공동 FE 시스템 전체의 관측 행 수와 상권 군집 수; 업종별 값을 합산한 상권 수가 아님',
    'areas_spanning_10_to_20': '동일 상권의 관측 최소 공연 ID 수≤10이고 최대≥20인 상권 수; 정확히 10·20을 관측한 쌍의 수가 아님',
    'total_store_size_group': '일반점포+프랜차이즈점포의 합 기준; 매출 모집단 일치 미검증이므로 점포당 매출 해석 제외',
    'source_id': 'input_integrity.json records의 source_id를 세미콜론으로 연결'}
notes = [
    '자료는 분기 자료이며 일별 분석이 아니다. 미제공 매출을 0으로 대체하지 않았다.',
    '업종마다 관측 상권·분기와 매출 제공률이 달라 직접적인 업종 경제규모 순위로 해석하지 않는다.',
    '상권 중첩 때문에 관측 매출 합은 중복이 제거된 자치구 총매출이 아니다.',
    '등록 공연 ID는 실제 상연 횟수·관객 수와 다르다. 공식 ID 연결과 시설명 잠정 연결은 구별한다.',
    '500m 분석에 기여한 12,839개 공연 ID는 공식 시설 ID 연결이다. 서울 전체 목록의 연결비율과 구별한다.',
    '현재 목록의 시설 좌표는 확보되었으나 과거 위치를 검증하지 않았다. 현재 상권 경계를 과거 분기에 적용했다.',
    '고정효과와 상권 군집 추론은 관측되지 않은 시간변화 요인, 상권 간 공간 상관, 매출 선택적 미제공을 모두 해결하지 않는다.',
    '비유의 결과는 효과 부재의 증거가 아니며, 기존 두 업종 결과 확인 뒤 10종으로 확대한 탐색 분석이다.']
schema = {'tables': {name: {'rows':len(frame),'columns':frame.columns.tolist(),
    'source_paths':source_map[name], 'sha256':sha256(OUT/name)} for name,frame in outputs.items()},
    'definitions':definitions, 'notes':notes}
(OUT/'evidence_schema.json').write_text(json.dumps(schema,ensure_ascii=False,indent=2),encoding='utf-8')

In [ ]:
for record in input_records.values():
    record['sha256_after'] = sha256(project_path(record['path']))
    record['unchanged'] = record['sha256_before'] == record['sha256_after']
assert all(row['unchanged'] for row in input_records.values())
integrity = {'created_utc':datetime.now(timezone.utc).isoformat(),
    'records':list(input_records.values()),
    'input_count':len(input_records),
    'metadata_hash_verified_count':sum(row['expected_sha256'] is not None for row in input_records.values()),
    'all_inputs_unchanged':True, 'outputs_count':len(outputs),
    'validation':{'source_coverage_equal':True,'store_coverage_equal':True,
        'contrast_transforms_equal':True,'main_sales_night_keys_equal':True,
        'raw_common_171_verified':True,'fe_common_170_verified':True,'no_new_model_fit':True}}
(OUT/'input_integrity.json').write_text(json.dumps(integrity,ensure_ascii=False,indent=2),encoding='utf-8')
print(f'완료: 근거표 {len(outputs)}개, 정의·범위·입력 검증 JSON 3개')
print(f'입력 {len(input_records)}개 실행 전후 동일 / 통계 메타데이터 해시 {integrity["metadata_hash_verified_count"]}개 일치')

**최종 해석:** 이 산출물은 기존 최종 분석을 읽기 쉬운 표로 옮긴 보고서 근거다. 매출 제공의 선택성과 업종별 표본 차이를 먼저 고려하고, 결과는 상권 안의 분기별 공연 활동과 매출 지표 사이의 조건부 관계로 설명한다. 유의성만으로 업종을 선택하거나 인과효과·투자효과·전체 지역 총매출 변화로 확대하지 않는다.